In [1]:
import sys
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.WARNING, force=True, stream=sys.stdout)
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.GradientDescent(model.parameters(), lr_method="lipschitz")
# opt = torch_numopt.GradientDescentLipschitz(model.parameters())
# opt = torch_numopt.ConjugateGradient(model.parameters())
# opt = torch_numopt.LBFGS(model.parameters())
# opt = torch_numopt.AdaHessian(model.parameters())
# opt = torch_numopt.GaussNewton(model.parameters())
opt = torch_numopt.LevenbergMarquardt(model.parameters())
# opt = torch_numopt.NewtonCG(model.parameters(), lr_init=1e-2)
# opt = torch_numopt.Newton(model.parameters(), lr_init=1e-3, lr_method="lipschitz")

# opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_method="bisect")
# opt = torch_numopt.ConjugateGradientLS(model.parameters(), line_search_cond="armijo", line_search_method="backtrack")
# opt = torch_numopt.LBFGSLS(model.parameters())
# opt = torch_numopt.AdaHessianLS(model.parameters())
# opt = torch_numopt.GaussNewtonLS(model.parameters())
# opt = torch_numopt.LevenbergMarquardtLS(model.parameters())
# opt = torch_numopt.NewtonLS(model.parameters(), solver="cg-trunc")
# opt = torch_numopt.NewtonLS(model.parameters())

# opt = torch_numopt.GradientDescentTR(model.parameters(), trust_region_method="cauchy")
# opt = torch_numopt.AdaHessianTR(model.parameters())
# opt = torch_numopt.GaussNewtonTR(model.parameters(), trust_region_method="dogleg")
# opt = torch_numopt.LevenbergMarquardtLS(model.parameters())
# opt = torch_numopt.NewtonTR(model.parameters(), trust_region_method="cauchy")

# opt = torch_numopt.NumericalOptimizer(
#     model.parameters(),
#     curvature_estimator=torch_numopt.ExactBlockHessianCalculator(),
#     lr_init=1e-2,
#     lr_method="scaled"
# )
# opt = torch_numopt.LineSearchOptimizer(
#     model.parameters(),
#     lr_init=1,
#     curvature_estimator=torch_numopt.ExactBlockHessianCalculator(),
#     line_search=torch_numopt.create_line_search_solver(
#         condition="armijo",
#         method="backtrack",
#         c1 = 1e-4,
#         c2 = 0.9,
#         tau = 0.1,
#         max_iter = 20,
#         tol = 1e-8,
#     ),
#     solver="solve"
# )
# curv_est = torch_numopt.ExactBlockHessianCalculator()
# opt = torch_numopt.TrustRegionOptimizer(
#     model.parameters(),
#     radius_init=1.0,
#     trust_region=torch_numopt.trust_region.ExactTRSolver(curv_est)
#     # trust_region=torch_numopt.trust_region.DoglegTRSolver(curv_est)
# )

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 59573301248.0
  prev_loss: 5.957330e+10, curr_loss: 1.982653e+26
  curr_step_dir: norm = 1.933083e+07
  curr_grad: norm = 9.460277e+10
  mu: 1.000000e-04
  lr: 1.000000e+00
  g_dot_p: -1.084774e+12
  p_norm_sq: 3.736808e+14
  pred_reduction: -5.610709e+11
  rho: 3.533694e+14
epoch:  1, loss: 1.9826527326283693e+26
  prev_loss: 1.982653e+26, curr_loss: inf
  curr_step_dir: norm = 5.742976e+13
  curr_grad: norm = inf
  mu: 1.000000e-05
  lr: 1.000000e+00
  g_dot_p: -2.906344e+27
  p_norm_sq: 3.298177e+27
  pred_reduction: -1.453189e+27
  rho: inf
epoch:  2, loss: inf


ValueError: NaN found in scaling matrix.

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -0.02090919017791748
Test metrics:  R2 = -0.03513002395629883
